 <a target="_blank" href="https://colab.research.google.com/github/Occhipinti-Lab/Workshop_Net4Brain/blob/main/Mini_Project_Spatial_notebook.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a> 
 
# Mini-Project: Multimodal Machine Learning on Glioblastoma Spatial Transcriptomics

**Net4Brain Training School**

Over three days you build a machine learning pipeline on human glioblastoma tissue. Work section by section: find the matching heading in `Demo_DLPFC.ipynb` and use the idea.

## Goal

Each tissue spot is either in the **tumour core** or in the **periphery**
surrounding it. Train a model to tell them apart from molecular data, then ask
which features it used and whether they make biological sense.

## Dataset

| **Tissue** | Human glioblastoma, IDH-wildtype |
|---|---|
| **Spots** | 35,187 tissue spots |
| **Patients** | 6 |
| **Sections** | 15 (2–5 per patient) |
| **Modality 1** | 2,000 highly variable genes — 10x Visium, **measured** |
| **Modality 2** | 168 metabolic module rates — scFEA, **inferred from the expression data** |
| **Label** | `Core` or `Periphery` |

## Change from the demo

| In the demo (DLPFC) | On your data (GBM) |
|---|---|
| `dlpfc_*.csv.gz` | `gbm_*.csv.gz` |
| `layer` | `region` |
| `donor` | `patient` |
| `sample_id` | `section` |
| `array_col`, `array_row` | `x`, `y` |
| Superficial / Deep | Core / Periphery |
| markers `MBP`, `MOBP`, `RELN` | markers `CHI3L1`, `VEGFA`, `NEFM` |

Your target is already in `df_meta["region"]`.

## How to use this notebook

Cells marked **TASK** are yours. Each one names the variables the rest of the
notebook expects. Do not delete the `# TODO` comments until you have replaced
them. If you get stuck, find the same section number in `Demo_DLPFC.ipynb` or ask tutors.

## A side note on the labels

`Periphery` is **not** healthy tissue. Glioblastoma infiltrates beyond its
visible core, so peripheral spots still contain tumour cells. The contrast is
*core versus surrounding tissue*, not *tumour versus normal*.

`Periphery` is also a merged class. The finer original annotation is in `region_detailed`.


> **This is the student notebook.** Cells marked **TASK** are for you to complete.
> The worked answers are in `Solution_GBM.ipynb` — try each task before looking.

# Day 1: Introduction & Preparing Data

**Goal today:** Exploratory data analysis and preprocessing - know what is in your data before you model anything.


## 1.1 Setup


In [1]:
# On Google Colab, uncomment the next line once.
# !pip install -q scanpy shap


In [1]:
import warnings
warnings.filterwarnings("ignore")
import os, urllib.request
import numpy as np
import pandas as pd
import scanpy as sc

sc.settings.verbosity = 1


## 1.2 Download the data

Three tables, all describing the same spots in the same row order. That shared
index is what makes this multimodal. This download is about 48 MB.


In [3]:
DATA_DIR = "dataset"
BASE_URL = "https://raw.githubusercontent.com/Occhipinti-Lab/Workshop_Net4Brain/main/dataset"

FILES = ["gbm_gene_expression.csv.gz",
         "gbm_flux.csv.gz",
         "gbm_metadata.csv.gz",
         "scfea_module_info.csv"]

os.makedirs(DATA_DIR, exist_ok=True)
for fname in FILES:
    dest = os.path.join(DATA_DIR, fname)
    if os.path.exists(dest):
        print(f"already present: {fname}")
        continue
    print(f"downloading {fname} ...", end=" ", flush=True)
    urllib.request.urlretrieve(f"{BASE_URL}/{fname}", dest)
    print(f"{os.path.getsize(dest)/1e6:.1f} MB")


already present: gbm_gene_expression.csv.gz
already present: gbm_flux.csv.gz
already present: gbm_metadata.csv.gz
already present: scfea_module_info.csv


## 1.3 TASK — load the three tables

Create `df_gene`, `df_flux`, `df_meta`. All three files are gzipped CSVs with
the spot ID in the first column, so they need `index_col=0`.

Then check that the three tables share the same row index. If the rows were in
different orders, every later result would be silently wrong.


In [ ]:
# TODO: load the three tables (see Demo 1.3; change dlpfc_ -> gbm_)
# Hint: pd.read_csv(f"{DATA_DIR}/<filename>", index_col=0)
df_gene = ...
df_flux = ...
df_meta = ...

print("gene expression :", df_gene.shape)
print("metabolic flux  :", df_flux.shape)
print("metadata        :", df_meta.shape)

# TODO: check that all three share the same row index
# Hint: df_gene.index.equals(df_meta.index)


### What is actually in each table?

Print the first few rows. Are the values on the scale you expect? Are there
obvious missing values? Do the column names look like gene symbols and module IDs?

Expression values are already **log-normalised** — do not run `sc.pp.normalize`
or `sc.pp.log1p`.


In [ ]:
# TODO: display the first few rows and columns of each table.
# Hint: Demo 1.3 — df.head() or df.iloc[:4, :6]


Your metadata columns:

- `region`: `Core` or `Periphery`. This is what you are predicting.
- `region_detailed`: the finer original annotation that `region` was merged from.
- `patient`: which of the 6 patients. This matters on Day 2.
- `section`: which of the 15 tissue sections.
- `x`, `y`: the spot's position in the tissue.


### Where the fluxomic data come from

The experiment is Visium, and the only molecule assayed at each spot is mRNA. The
flux table was computed from the gene expression by
[scFEA](https://github.com/changwn/scFEA), which we ran before the workshop.

Human central metabolism is written as a graph. scFEA collapses it into **168
modules** and predicts one flux per module per spot from the genes encoding
those enzymes. Say **"inferred flux"**, never "measured flux". A gene and a
flux module appearing together near the top of a ranking is one piece of
evidence, not two.


## 1.4 TASK — pack the tables into AnnData

Copy the pattern from Demo 1.4. The only changes are the coordinate columns:
use `x` and `y` instead of `array_col` and `array_row`, and flip `y` so the
tissue is not upside-down.


In [ ]:
# TODO: build adata (genes) and adata_flux (modules), sharing the same obs.
# Hint: Demo 1.4
#   adata = sc.AnnData(df_gene, obs=df_meta.copy())
#   adata.obsm["spatial"] = np.column_stack([adata.obs["x"], -adata.obs["y"]])
adata = ...
adata_flux = ...

print(adata)
print(adata_flux)


## 1.5 Exploratory data analysis


### TASK — how much data do you have?

Bar plots of spots per region, spots per patient, and sections per patient.


In [ ]:
# TODO: three bar plots (Demo 1.5)
# adata.obs["region"].value_counts().plot.bar(...)
# adata.obs["patient"].value_counts().plot.bar(...)
# adata.obs.groupby("patient")["section"].nunique().plot.bar(...)


### TASK — what does the tissue look like?

Plot one section, coloured by `region` (Demo 1.5).


In [ ]:
# TODO: spatial plot of one section (Demo 1.5)
# sec = adata.obs["section"].value_counts().index[0]
# sub = adata[adata.obs["section"] == sec]
# sc.pl.embedding(sub, basis="spatial", color="region")


### TASK — marker genes

Violin and spatial plots of `CHI3L1`, `VEGFA`, `NEFM` (Demo 1.5).
`CHI3L1` and `VEGFA` should be higher in Core; `NEFM` in Periphery.


In [ ]:
# TODO: violin + spatial plots of ["CHI3L1", "VEGFA", "NEFM"] (Demo 1.5)


### TASK — metabolic modules

Pick a few module IDs and plot them the same way as the genes (Demo 1.5).


In [ ]:
# TODO: violin + spatial plots of a few modules, e.g. ["M_2", "M_3", "M_25"]
# Hint: load scfea_module_info.csv if you want the reaction names


## 1.6 TASK — labels

Your label is already in `region`. Set `y` and `groups` (Demo 1.6, without the layer merge).


In [ ]:
# TODO:
# y = adata.obs["region"]
# groups = adata.obs["patient"]
y = ...
groups = ...


---
# Day 2: Training models



In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay


## 2.1 TASK — train / test split

Hold out one whole patient, as in Demo 2.1.


In [ ]:
# TODO: hold out one patient (Demo 2.1)
# held_out = sorted(groups.unique())[0]
# train = groups != held_out
# test = groups == held_out
held_out = ...
train = ...
test = ...


## 2.2 TASK — train a classifier

Scale, fit `LogisticRegression` on `df_gene`, predict the test spots (Demo 2.2).


In [ ]:
# TODO: scaler, model, fit, predict (Demo 2.2)
# Keep pred — the next cell uses it.
pred = ...


## 2.3 TASK — classification results

Accuracy, classification report, and confusion matrix (Demo 2.3).


In [ ]:
# TODO:
# print(accuracy_score(y[test], pred))
# print(classification_report(y[test], pred))
# ConfusionMatrixDisplay.from_predictions(y[test], pred, cmap="Blues")


## 2.4 TASK — compare modalities

Train the same classifier on genes, flux, and both (Demo 2.4). Then fit the fused model and keep it for Day 3.


In [ ]:
# TODO: three accuracies + bar plot, then fit the fused model (Demo 2.4)
# Keep model, X_train, X_test, feat_names for Day 3.


---
# Day 3: Explainable AI


## 3.1 TASK — SHAP

The default SHAP summary (beeswarm) plot. See Demo 3.1.


In [ ]:
import shap

# TODO: LinearExplainer on X_train, then shap.summary_plot on a subset of X_test (Demo 3.1)
# explainer = shap.LinearExplainer(model, X_train)
# shap_values = explainer.shap_values(X_test[:1000])
# shap.summary_plot(shap_values, X_test[:1000], feature_names=list(feat_names))

## 3.2 TASK — logistic coefficients

The 15 largest coefficients, as a table and a bar plot (Demo 3.2).


In [ ]:
# TODO: pd.Series(model.coef_[0], index=feat_names)  (Demo 3.2)


## How the flux values were generated

The flux tables are **not measured**. They were computed from the gene-expression
counts with [scFEA](https://github.com/changwn/scFEA) before the workshop. You
do not need to run this. It is here so you can see how the second modality
was produced.

```bash
git clone https://github.com/changwn/scFEA.git

# input: a genes x spots raw count matrix as CSV
python src/scFEA.py \
  --input_dir  <dir containing the count matrix> \
  --test_file  <counts.csv> \
  --moduleGene_file       module_gene_m168.csv \
  --stoichiometry_matrix  cmMat_c70_m168.csv \
  --cName_file            cName_c70_m168.csv \
  --sc_imputation True \
  --train_epoch 30 \
  --output_flux_file      <flux.csv>
```

`module_gene_m168.csv` maps genes to modules; `cmMat_c70_m168.csv` is the
stoichiometry matrix (70 compounds × 168 modules). The human M168 module set
was used for both datasets.
